> **Versión para estudiante.** Las celdas marcadas con `# TODO` están incompletas: complete el código siguiendo las pistas. La solución completa está en `notebooks-soluciones/sesion-04-preprocesamiento-de-datos.ipynb`.


# Sesión 4 · Preprocesamiento de datos
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

**Objetivo:** tomar un conjunto de datos "sucio" de una estación solar y eólica y dejarlo listo para entrenar un modelo, paso a paso.

**Ruta:** 1) crear datos · 2) primer vistazo · 3) tipos y valores raros · 4) duplicados · 5) atípicos · 6) faltantes · 7) variables nuevas · 8) división · 9) escalado.

> Ejecute cada celda con `Shift + Enter` y lea qué hace antes de pasar a la siguiente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
pd.set_option('display.max_columns', 20)

## 1. Los datos
Para que el cuaderno funcione sin descargar nada, **simulamos** 30 días de datos por hora con problemas típicos de la vida real. Al final, cambie esta parte por un CSV real (datos.gov.co, IDEAM o Kaggle) con `pd.read_csv('archivo.csv')`.

In [ ]:
rng = np.random.default_rng(42)
idx = pd.date_range('2025-01-01', periods=24*30, freq='h')
hora = idx.hour
irr = np.clip(900*np.sin(np.pi*(hora-6)/12), 0, None) * rng.uniform(0.5, 1, len(idx))
temp = 18 + 8*np.sin(np.pi*(hora-9)/12) + rng.normal(0, 1, len(idx))
viento = np.abs(5 + 2*np.sin(idx.dayofyear/3) + rng.normal(0, 1.5, len(idx)))
gen = 0.25*irr + 10*viento + rng.normal(0, 15, len(idx))
df = pd.DataFrame({'fecha_hora': idx.strftime('%Y-%m-%d %H:%M'),
                   'irradiancia_wm2': irr.round(1), 'temperatura_c': temp.round(1),
                   'velocidad_viento_ms': viento.round(2), 'cielo': rng.choice(['Despejado','Parcial','Nublado'], len(idx)),
                   'generacion_kw': gen.round(1)})
# --- Ensuciamos los datos a propósito ---
df.loc[rng.choice(len(df), 40, replace=False), 'irradiancia_wm2'] = np.nan   # faltantes
df.loc[rng.choice(len(df), 25, replace=False), 'temperatura_c'] = -999       # valor centinela
df.loc[rng.choice(len(df), 5, replace=False), 'velocidad_viento_ms'] *= 10   # atípicos
df['velocidad_viento_ms'] = df['velocidad_viento_ms'].astype(str).str.replace('.', ',', regex=False)  # coma decimal
df.loc[::7, 'cielo'] = df.loc[::7, 'cielo'].str.upper() + ' '                # texto inconsistente
df = pd.concat([df, df.iloc[100:110]], ignore_index=True)                    # filas duplicadas
df.head()

## 2. Primer vistazo
Antes de cambiar nada, **mire** los datos: tamaño, tipos y estadísticas. Pregunta guía: *¿algo parece raro?*

In [ ]:
print(df.shape)
df.info()
df.describe()

**Qué buscar:** `velocidad_viento_ms` aparece como texto (`object`) y no como número; `temperatura_c` tiene un mínimo de -999, imposible para una temperatura.

## 3. Tipos de datos y valores raros
- El **-999** es un *valor centinela*: alguien lo usó para decir "sin dato". Lo convertimos en `NaN` (faltante).
- La coma decimal hay que cambiarla por punto para poder convertir a número.
- La fecha debe ser fecha real, no texto.
- El texto debe escribirse siempre igual.

In [ ]:
# TODO: Limpieza de tipos y valores centinela
# 1) Reemplace el valor centinela -999 en 'temperatura_c' por un valor faltante real.
#    Pista: Series.replace(-999, np.nan)
# 2) 'velocidad_viento_ms' viene como texto con coma decimal (ej. '3,45').
#    Cambie la coma por punto y conviértala a número.
#    Pista: str.replace(',', '.') + pd.to_numeric(...)
# 3) Convierta 'fecha_hora' a tipo fecha real.
#    Pista: pd.to_datetime(...)
# 4) Normalice el texto de 'cielo': quite espacios y pase a minúsculas.
#    Pista: str.strip().str.lower()
# 5) Muestre df['cielo'].value_counts() y df.dtypes para verificar.

# Su código aquí


## 4. Duplicados
Filas repetidas hacen que el modelo "vea dos veces" lo mismo.

In [ ]:
print('Duplicados:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Filas después:', len(df))

## 5. Atípicos (valores extremos)
Usamos la **regla del rango intercuartílico (IQR)**: es atípico lo que queda muy por fuera de la caja del diagrama de cajas.

In [ ]:
# TODO: Detección y tratamiento de atípicos con la regla del IQR
# 1) Calcule el primer y tercer cuartil (Q1, Q3) de 'velocidad_viento_ms'.
#    Pista: df[col].quantile([0.25, 0.75])
# 2) Calcule el límite superior: Q3 + 1.5 * (Q3 - Q1)
# 3) Cuente e imprima cuántos valores superan ese límite.
# 4) Grafique un boxplot antes de tratar los atípicos.
#    Pista: df[col].plot(kind='box', vert=False); plt.show()
# 5) Marque como NaN (faltante) los valores por encima del límite superior.
#    Pista: df.loc[df[col] > lim_sup, col] = np.nan

# Su código aquí


> **Ojo:** un atípico no siempre es un error. Un pico real de viento puede ser justo lo que queremos predecir. Aquí lo marcamos como dudoso porque un viento de 50 m/s es imposible en este sensor.

## 6. Datos faltantes
Primero medimos cuántos faltan; después decidimos cómo rellenar. En **series de tiempo** lo natural es **interpolar** usando las horas vecinas, no usar el promedio global.

In [ ]:
# TODO: Detección e imputación de valores faltantes
# 1) Mida el porcentaje de valores faltantes por columna.
#    Pista: df.isna().mean() * 100
# 2) Ordene el DataFrame por 'fecha_hora' y use esa columna como índice.
#    Pista: df.sort_values('fecha_hora').set_index('fecha_hora')
# 3) Para las columnas numéricas ('irradiancia_wm2', 'temperatura_c',
#    'velocidad_viento_ms'), interpole usando el tiempo como referencia,
#    y rellene los extremos que queden sin interpolar.
#    Pista: df[num].interpolate(method='time').bfill().ffill()
# 4) Verifique que ya no queden faltantes.
#    Pista: df.isna().sum().sum()
# 5) (Opcional) Grafique 'irradiancia_wm2' antes y después de interpolar
#    para un rango de fechas, ej. '2025-01-05':'2025-01-07'.

# Su código aquí


## 7. Variables nuevas (ingeniería de características)
- `hora` y `dia_semana` salen de la fecha: la generación solar depende mucho de la hora.
- El texto `cielo` se convierte en columnas 0/1 (*one-hot*), porque los modelos solo entienden números.

In [ ]:
# TODO: Ingeniería de características y codificación categórica
# 1) Cree columnas nuevas 'hora' y 'dia_semana' a partir del índice de fecha.
#    Pista: df.index.hour , df.index.dayofweek
# 2) Codifique la variable categórica 'cielo' con one-hot encoding
#    (una columna 0/1 por cada categoría).
#    Pista: pd.get_dummies(df, columns=['cielo'], dtype=int)
# 3) Muestre las primeras filas del resultado con df.head()

# Su código aquí


## 8. División entrenamiento / prueba
En series de tiempo **no se mezclan** las filas: se entrena con el pasado (80 %) y se prueba con el futuro (20 %).

In [ ]:
corte = int(len(df)*0.8)
X = df.drop(columns='generacion_kw'); y = df['generacion_kw']
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

## 9. Escalado
Variables con rangos muy distintos (irradiancia hasta 900, viento hasta 10) confunden a algunos modelos. El escalador se **ajusta solo con el entrenamiento** y luego se aplica a la prueba; si no, estaríamos "espiando" el futuro (*fuga de datos*).

In [ ]:
# TODO: Escalado de variables numéricas (sin fuga de datos)
# 1) Defina las columnas numéricas a escalar:
#    ['irradiancia_wm2', 'temperatura_c', 'velocidad_viento_ms']
# 2) Ajuste (fit) un StandardScaler SOLO con X_train (nunca con X_test).
#    Pista: StandardScaler().fit(X_train[cols])
# 3) Transforme tanto X_train como X_test con ese mismo escalador.
#    Pista: esc.transform(X_train[cols]) / esc.transform(X_test[cols])
# 4) Muestre X_train_s[cols].describe().round(2) para verificar
#    que la media quede cerca de 0 y la desviación estándar cerca de 1.

# Su código aquí


## Resumen y ejercicios
**Checklist:** tipos correctos ✔ · centinelas ✔ · duplicados ✔ · atípicos ✔ · faltantes ✔ · variables nuevas ✔ · división en el tiempo ✔ · escalado sin fuga ✔

**Ejercicios**
1. Repita el tratamiento de atípicos para `temperatura_c`. ¿Hay alguno?
2. Cambie `interpolate` por rellenar con la mediana. ¿Cómo cambia la gráfica de la sección 6?
3. Pregunte a un asistente de IA: *"¿Qué diferencia hay entre StandardScaler y MinMaxScaler?"* y pruebe `MinMaxScaler` aquí.
4. Cargue un CSV real y aplique la misma ruta, paso a paso.